# Tweede Kamer Data Analysis

This notebook explores and analyzes data from the `tweedekamer.duckdb` file.

In [ ]:
# Install DuckDB library
%pip install duckdb

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    # Fallback: check known volume paths in Databricks
    fallback_paths = [
        Path("/Volumes/workspace/tweede-kamer/data_files/tweedekamer.duckdb"),
    ]
    for candidate in fallback_paths:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")

database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)

def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()

database_path

In [ ]:
# List silver tables in the DuckDB database
tables = connection.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
      AND table_schema = 'silver'
    ORDER BY table_name
""").fetchall()
print(f"Found {len(tables)} silver tables:\n")
for schema, name in tables:
    print(f"  - {schema}.{name}")

In [ ]:
# Get schema information for each table
for schema_name, table_name in tables:
    full_name = f"{schema_name}.{table_name}"
    print(f"\n{'='*60}")
    print(f"Table: {full_name}")
    print(f"{'='*60}")
    
    # Get column info
    schema = connection.execute(f"DESCRIBE {full_name}").fetchdf()
    print(schema.to_string(index=False))
    
    # Get row count
    row_count = connection.execute(f"SELECT COUNT(*) FROM {full_name}").fetchone()[0]
    print(f"\nRow count: {row_count:,}")

## Preview Data from Each Table

Let's look at a sample of data from each table.

In [ ]:
# Preview first 5 rows of each table
for schema_name, table_name in tables:
    full_name = f"{schema_name}.{table_name}"
    print(f"\n{'='*60}")
    print(f"Preview: {full_name}")
    print(f"{'='*60}")
    
    df = connection.execute(f"SELECT * FROM {full_name} LIMIT 5").fetchdf()
    display(df)

## Convert to Spark DataFrames

Convert DuckDB tables to Spark DataFrames for analysis using Databricks capabilities.

In [ ]:
# Convert all tables to Spark DataFrames and store in a dictionary
spark_tables = {}

for schema_name, table_name in tables:
    full_name = f"{schema_name}.{table_name}"
    print(f"Converting {full_name} to Spark DataFrame...")
    
    # Read from DuckDB
    pandas_df = connection.execute(f"SELECT * FROM {full_name}").fetchdf()
    
    # Convert to Spark
    spark_df = spark.createDataFrame(pandas_df)
    spark_tables[full_name] = spark_df
    
    print(f"  ✅ {full_name}: {len(pandas_df):,} rows, {len(pandas_df.columns)} columns")

print(f"\n✅ All {len(spark_tables)} tables converted to Spark DataFrames")

## Optional: Save to Delta Tables

Uncomment and run the cell below to save all tables as Delta tables in the `tweede-kamer` schema for better performance and persistence.

In [ ]:
# Save all tables as Delta tables
# Uncomment the code below to save

# for table_name, spark_df in spark_tables.items():
#     print(f"Saving {table_name} as Delta table...")
#     spark_df.write.mode("overwrite").saveAsTable(f"workspace.`tweede-kamer`.{table_name}")
#     print(f"  ✅ Saved as workspace.tweede-kamer.{table_name}")
# 
# print(f"\n✅ All tables saved to workspace.tweede-kamer schema")

## Start Your Analysis Below

Use the `spark_tables` dictionary to access Spark DataFrames:
```python
# Example:
df = spark_tables['table_name']
df.show()
```

Or query directly from DuckDB:
```python
result = connection.execute("SELECT * FROM table_name WHERE ...").fetchdf()
display(result)
```

In [ ]:
# People who have been in multiple fracties, showing all their distinct fracties
# PvdA, GL, GroenLinks-PvdA and PRO are treated as the same fractie under the name PRO
result = query_df("""
WITH normalized AS (
    SELECT
        p.id AS persoon_id,
        p.roepnaam, p.voornamen, p.tussenvoegsel, p.achternaam,
        CASE
            WHEN f.afkorting IN ('PvdA', 'GL', 'GroenLinks-PvdA', 'PRO') THEN 'PRO'
            ELSE f.afkorting
        END AS fractie_norm
    FROM silver.persoon p
    JOIN silver.fractie_zetel_persoon fzp ON fzp.persoon_id = p.id
    JOIN silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = fz.fractie_id
),
fractie_count AS (
    SELECT
        persoon_id,
        COUNT(DISTINCT fractie_norm) AS n_fracties
    FROM normalized
    GROUP BY persoon_id
    HAVING COUNT(DISTINCT fractie_norm) > 1
)
SELECT
    n.persoon_id,
    COALESCE(n.roepnaam, n.voornamen, '') 
        || COALESCE(' ' || NULLIF(n.tussenvoegsel, ''), '') 
        || ' ' || n.achternaam AS naam,
    string_agg(DISTINCT n.fractie_norm, ', ') AS fractie_afkortingen,
    COUNT(DISTINCT n.fractie_norm) AS aantal_fracties
FROM normalized n
WHERE n.persoon_id IN (SELECT persoon_id FROM fractie_count)
GROUP BY n.persoon_id, n.roepnaam, n.voornamen, n.tussenvoegsel, n.achternaam
ORDER BY n.achternaam
""")
print(f"Found {len(result)} people who switched fracties:")
display(result)

#Fractielid wisselingen
Deze grafiek laat ziet hoeveel leden verschillende fracties gemeen hebben gehad door de jaren heen. Mona Keijzer is bijvoorbeeld lid geweest can BBB, CDA en Lid Keijzer. Daarom zie je tussen de bolletjes van die 3 partijen lijntjes staan met 1. 
De grafiek is gesorteerd en gegroepeerd op basis van veel voorkomende overeenkomsten tussen partijen. Deze groeperingen kunnen op meerdere manieren verklaard worden, bijvoorbeeld doordat partijen veel gelijke standpunten hebben, of omdat er bij een afsplitsingspartij veel leden over zijn gestapt.

In [ ]:
# Build a network graph of fracties connected by people who switched between them
# PvdA, GL, GroenLinks-PvdA and PRO are treated as the same fractie under the name PRO
# Communities are detected and color-coded to group tightly connected fracties together
import itertools

%pip install networkx -q
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

# Get each person's distinct fracties (normalized)
persoon_fracties = query_df("""
WITH normalized AS (
    SELECT DISTINCT
        fzp.persoon_id,
        CASE
            WHEN f.afkorting IN ('PvdA', 'GL', 'GroenLinks-PvdA', 'PRO') THEN 'PRO'
            ELSE f.afkorting
        END AS fractie
    FROM silver.fractie_zetel_persoon fzp
    JOIN silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = fz.fractie_id
)
SELECT persoon_id, string_agg(fractie, '|') AS fracties
FROM normalized
GROUP BY persoon_id
""")

# Build edge weights: for each person with >1 fractie, connect all pairs
edge_weights = {}
for row in persoon_fracties.itertuples(index=False):
    fracties = sorted(set(row.fracties.split('|')))
    if len(fracties) > 1:
        for a, b in itertools.combinations(fracties, 2):
            edge_weights[(a, b)] = edge_weights.get((a, b), 0) + 1

# Build graph
G = nx.Graph()
for (a, b), weight in edge_weights.items():
    G.add_edge(a, b, weight=weight)

# Detect communities (clusters of tightly connected fracties)
communities = list(greedy_modularity_communities(G, weight='weight'))

color_map = plt.cm.Set3
community_colors = {}
for i, comm in enumerate(communities):
    for node in comm:
        community_colors[node] = color_map(i / max(len(communities), 1))

# Use Kamada-Kawai layout: pulls strongly connected nodes closer together
pos = nx.kamada_kawai_layout(G, weight='weight')

# Draw
fig, ax = plt.subplots(figsize=(16, 11))

edges = G.edges(data=True)
weights = [d['weight'] for _, _, d in edges]
max_w = max(weights) if weights else 1
edge_widths = [1 + 5 * (w / max_w) for w in weights]

degrees = dict(G.degree(weight='weight'))
node_sizes = [300 + 1500 * (degrees[n] / max(degrees.values())) for n in G.nodes]
node_colors = [community_colors[n] for n in G.nodes]

nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.4, edge_color='steelblue', ax=ax)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, edgecolors='darkgray', linewidths=1.5, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)

# Edge labels showing switcher counts
edge_labels = {(a, b): d['weight'] for a, b, d in edges}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, font_color='darkblue', ax=ax)

# Legend for communities
for i, comm in enumerate(communities):
    ax.scatter([], [], c=[color_map(i / max(len(communities), 1))], s=100, edgecolors='darkgray',
               label=f"Cluster {i+1}: {', '.join(sorted(comm))}")
ax.legend(loc='upper left', fontsize=7, framealpha=0.9, title='Gedetecteerde clusters')

ax.set_title('Fractie Network: clusters van fracties die leden delen', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\n{len(G.nodes)} fracties, {len(G.edges)} verbindingen, {len(communities)} clusters")
for i, comm in enumerate(communities):
    print(f"  Cluster {i+1} ({len(comm)} fracties): {', '.join(sorted(comm))}")
print(f"\nTop 10 sterkste verbindingen:")
for a, b, w in sorted(G.edges(data=True), key=lambda x: -x[2]['weight'])[:10]:
    print(f"  {a} <-> {b}: {w['weight']} personen")

#Totaal aantal zetelwisselingen tegenover Gemiddeld aantal zetelwisselingen per jaar sinds bestaan.
Hier is te zien dat over het algemeen racties die langer bestaan meer zetelwisselingen hebben gehad totaal, maar het aantal zetewisselingen sinds de fractie bestaat zit geen patroon in.

In [ ]:
# Bar chart: zetelwisselingen (seat changes) per fractie, 2020–2026
# Counts both arrivals (van in range) and departures (tot_en_met in range)
# Only fracties that were active between 2020 and 2026 are included
# PvdA, GL, GroenLinks-PvdA and PRO are normalized to PRO

zetelwisselingen = query_df("""
WITH active_fracties AS (
    SELECT
        id,
        afkorting,
        naam_nl,
        datum_actief,
        datum_inactief,
        CASE
            WHEN afkorting IN ('PvdA', 'GL', 'GroenLinks-PvdA', 'PRO') THEN 'PRO'
            ELSE afkorting
        END AS fractie_norm
    FROM silver.fractie
    WHERE datum_actief <= '2026-12-31'
      AND (datum_inactief IS NULL OR datum_inactief >= '2020-01-01')
),
seat_events AS (
    -- Arrivals: person joins a fractie
    SELECT
        f.fractie_norm,
        fzp.persoon_id,
        'Aanmelding' AS event_type,
        fzp.van AS event_date
    FROM silver.fractie_zetel_persoon fzp
    JOIN silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
    JOIN active_fracties f ON f.id = fz.fractie_id
    WHERE fzp.van >= DATE '2020-01-01'
      AND fzp.van <= DATE '2026-12-31'

    UNION ALL

    -- Departures: person leaves a fractie
    SELECT
        f.fractie_norm,
        fzp.persoon_id,
        'Vertrek' AS event_type,
        fzp.tot_en_met AS event_date
    FROM silver.fractie_zetel_persoon fzp
    JOIN silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
    JOIN active_fracties f ON f.id = fz.fractie_id
    WHERE fzp.tot_en_met IS NOT NULL
      AND fzp.tot_en_met >= DATE '2020-01-01'
      AND fzp.tot_en_met <= DATE '2026-12-31'
),
fractie_years AS (
    SELECT
        fractie_norm,
        SUM(
            GREATEST(
                DATE_DIFF('day', datum_actief, COALESCE(datum_inactief, DATE '2026-10-10')) / 365.25,
                0.01
            )
        ) AS jaren_bestaan
    FROM active_fracties
    GROUP BY fractie_norm
)
SELECT
    se.fractie_norm AS fractie,
    COUNT(*) AS zetelwisselingen,
    SUM(CASE WHEN event_type = 'Aanmelding' THEN 1 ELSE 0 END) AS aanmeldingen,
    SUM(CASE WHEN event_type = 'Vertrek' THEN 1 ELSE 0 END) AS vertrekken,
    ROUND(fy.jaren_bestaan, 1) AS jaren_bestaan
FROM seat_events se
JOIN fractie_years fy ON fy.fractie_norm = se.fractie_norm
GROUP BY se.fractie_norm, fy.jaren_bestaan
ORDER BY fy.jaren_bestaan DESC
""")

print(f"Found {len(zetelwisselingen)} fracties with zetelwisselingen (2020–2026):")
display(zetelwisselingen)

# Plot stacked bar chart
fig, ax = plt.subplots(figsize=(14, 7))

x = range(len(zetelwisselingen))
bar_width = 0.6

bars_aan = ax.bar(x, zetelwisselingen['aanmeldingen'], bar_width, label='Aanmeldingen', color='#4CAF50')
bar_vertrek = ax.bar(x, zetelwisselingen['vertrekken'], bar_width,
                      bottom=zetelwisselingen['aanmeldingen'], label='Vertrekken', color='#EF5350')

ax.set_xlabel('Fractie', fontsize=12)
ax.set_ylabel('Aantal zetelwisselingen', fontsize=12)
ax.set_title('Zetelwisselingen per fractie (2020–2026)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(zetelwisselingen['fractie'], rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=11)

# Add total count labels on top of each bar
for i, total in enumerate(zetelwisselingen['zetelwisselingen']):
    ax.text(i, total + 0.3, str(total), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: zetelwisselingen per jaar van bestaan (entire existence, normalized)
# Same fracties as previous chart (active 2020–2026), PRO normalization applied
# Total seat changes (all-time) divided by years the fractie existed

wisselingen_per_jaar = query_df("""
WITH active_fracties AS (
    SELECT
        id,
        afkorting,
        naam_nl,
        datum_actief,
        datum_inactief,
        CASE
            WHEN afkorting IN ('PvdA', 'GL', 'GroenLinks-PvdA', 'PRO') THEN 'PRO'
            ELSE afkorting
        END AS fractie_norm
    FROM silver.fractie
    WHERE datum_actief <= '2026-12-31'
      AND (datum_inactief IS NULL OR datum_inactief >= '2020-01-01')
),
-- Years of existence per normalized fractie (sum across constituent records)
fractie_years AS (
    SELECT
        fractie_norm,
        SUM(
            GREATEST(
                DATE_DIFF('day', datum_actief, COALESCE(datum_inactief, DATE '2026-10-10')) / 365.25,
                0.01
            )
        ) AS jaren_bestaan
    FROM active_fracties
    GROUP BY fractie_norm
),
-- All seat events across entire existence
seat_events AS (
    SELECT
        f.fractie_norm,
        'Aanmelding' AS event_type
    FROM silver.fractie_zetel_persoon fzp
    JOIN silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
    JOIN active_fracties f ON f.id = fz.fractie_id
    WHERE fzp.van IS NOT NULL

    UNION ALL

    SELECT
        f.fractie_norm,
        'Vertrek' AS event_type
    FROM silver.fractie_zetel_persoon fzp
    JOIN silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
    JOIN active_fracties f ON f.id = fz.fractie_id
    WHERE fzp.tot_en_met IS NOT NULL
),
event_counts AS (
    SELECT
        fractie_norm,
        COUNT(*) AS zetelwisselingen,
        SUM(CASE WHEN event_type = 'Aanmelding' THEN 1 ELSE 0 END) AS aanmeldingen,
        SUM(CASE WHEN event_type = 'Vertrek' THEN 1 ELSE 0 END) AS vertrekken
    FROM seat_events
    GROUP BY fractie_norm
)
SELECT
    ec.fractie_norm AS fractie,
    ec.zetelwisselingen,
    ec.aanmeldingen,
    ec.vertrekken,
    ROUND(fy.jaren_bestaan, 1) AS jaren_bestaan,
    ROUND(ec.zetelwisselingen / fy.jaren_bestaan, 2) AS wisselingen_per_jaar
FROM event_counts ec
JOIN fractie_years fy ON fy.fractie_norm = ec.fractie_norm
ORDER BY fy.jaren_bestaan DESC
""")

print(f"Found {len(wisselingen_per_jaar)} fracties:")
display(wisselingen_per_jaar)

# Plot bar chart
fig, ax = plt.subplots(figsize=(14, 7))

x = range(len(wisselingen_per_jaar))
bar_width = 0.6

bars = ax.bar(x, wisselingen_per_jaar['wisselingen_per_jaar'], bar_width, color='#42A5F5', edgecolor='#1565C0', linewidth=0.5)

ax.set_xlabel('Fractie', fontsize=12)
ax.set_ylabel('Zetelwisselingen per jaar', fontsize=12)
ax.set_title('Zetelwisselingen per jaar van bestaan (gehele bestaansperiode)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(wisselingen_per_jaar['fractie'], rotation=45, ha='right', fontsize=10)

# Add value labels on top of each bar
for i, val in enumerate(wisselingen_per_jaar['wisselingen_per_jaar']):
    ax.text(i, val + 0.1, f'{val:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

#Totaal zetels tegenover gemiddelde zetelwisseling.
Dit is moeilijk weer te geven omdat de meeste zetelwisselingen bij fracties gebeuren bij de verkiezingen. Verkiezingen gebeuren meestal niet elk jaar, maar toch is er wel een visualisatie gemaakt op jaar gemiddelde.

In [ ]:
# Bar chart: ratio between average zetels and average zetelwisselingen per jaar (2020–2026)
# Avg zetels = average number of people holding a seat per year (2020–2026)
# Avg wisselingen = total seat changes / number of active years in range
# Only fracties active between 2020 and 2026, with PRO normalization

zetel_ratio = query_df("""
WITH active_fracties AS (
    SELECT
        id,
        afkorting,
        CASE
            WHEN afkorting IN ('PvdA', 'GL', 'GroenLinks-PvdA', 'PRO') THEN 'PRO'
            ELSE afkorting
        END AS fractie_norm
    FROM silver.fractie
    WHERE datum_actief <= '2026-12-31'
      AND (datum_inactief IS NULL OR datum_inactief >= '2020-01-01')
),
-- Average zetels: count active members per year, then average across years
years AS (
    SELECT * FROM (VALUES (2020), (2021), (2022), (2023), (2024), (2025), (2026)) AS t(year_val)
),
yearly_zetels AS (
    SELECT
        f.fractie_norm,
        y.year_val,
        COUNT(DISTINCT fzp.persoon_id) AS n_zetels
    FROM active_fracties f
    JOIN silver.fractie_zetel fz ON fz.fractie_id = f.id
    JOIN silver.fractie_zetel_persoon fzp ON fzp.fractie_zetel_id = fz.id
    CROSS JOIN years y
    WHERE EXTRACT(YEAR FROM fzp.van) <= y.year_val
      AND (fzp.tot_en_met IS NULL OR EXTRACT(YEAR FROM fzp.tot_en_met) >= y.year_val)
    GROUP BY f.fractie_norm, y.year_val
),
avg_zetels AS (
    SELECT
        fractie_norm,
        AVG(n_zetels) AS avg_zetels,
        COUNT(*) AS n_active_years
    FROM yearly_zetels
    GROUP BY fractie_norm
),
-- Total zetelwisselingen 2020–2026
seat_events AS (
    SELECT f.fractie_norm
    FROM silver.fractie_zetel_persoon fzp
    JOIN silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
    JOIN active_fracties f ON f.id = fz.fractie_id
    WHERE fzp.van >= DATE '2020-01-01' AND fzp.van <= DATE '2026-12-31'

    UNION ALL

    SELECT f.fractie_norm
    FROM silver.fractie_zetel_persoon fzp
    JOIN silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
    JOIN active_fracties f ON f.id = fz.fractie_id
    WHERE fzp.tot_en_met IS NOT NULL
      AND fzp.tot_en_met >= DATE '2020-01-01'
      AND fzp.tot_en_met <= DATE '2026-12-31'
),
event_counts AS (
    SELECT fractie_norm, COUNT(*) AS zetelwisselingen
    FROM seat_events
    GROUP BY fractie_norm
)
SELECT
    az.fractie_norm AS fractie,
    ROUND(az.avg_zetels, 1) AS avg_zetels,
    az.n_active_years AS actieve_jaren,
    ROUND(COALESCE(ec.zetelwisselingen, 0) * 1.0 / az.n_active_years, 1) AS avg_wisselingen_per_jaar,
    ROUND(COALESCE(ec.zetelwisselingen, 0) * 1.0 / az.n_active_years / NULLIF(az.avg_zetels, 0), 2) AS ratio
FROM avg_zetels az
LEFT JOIN event_counts ec ON ec.fractie_norm = az.fractie_norm
WHERE az.avg_zetels > 0
ORDER BY ratio DESC
""")

print(f"Found {len(zetel_ratio)} fracties:")
display(zetel_ratio)

# Plot grouped bar chart with ratio annotations
fig, ax1 = plt.subplots(figsize=(15, 8))

x = list(range(len(zetel_ratio)))
bar_width = 0.35

bars1 = ax1.bar([i - bar_width/2 for i in x], zetel_ratio['avg_zetels'], bar_width,
                label='Gemiddeld aantal zetels', color='#66BB6A', edgecolor='#2E7D32', linewidth=0.5)
bars2 = ax1.bar([i + bar_width/2 for i in x], zetel_ratio['avg_wisselingen_per_jaar'], bar_width,
                label='Gemiddelde wisselingen per jaar', color='#EF5350', edgecolor='#C62828', linewidth=0.5)

ax1.set_xlabel('Fractie', fontsize=12)
ax1.set_ylabel('Aantal', fontsize=12)
ax1.set_title('Gemiddelde zetels vs gemiddelde zetelwisselingen per jaar (2020–2026)', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(zetel_ratio['fractie'], rotation=45, ha='right', fontsize=9)
ax1.legend(loc='upper right', fontsize=10)

# Annotate ratio above each pair of bars
for i, row in zetel_ratio.iterrows():
    max_h = max(row['avg_zetels'], row['avg_wisselingen_per_jaar'])
    ax1.text(i, max_h + 0.5, f"{row['ratio']:.1f}x",
             ha='center', va='bottom', fontsize=8, fontweight='bold',
             color='#1565C0',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='#E3F2FD', edgecolor='#1565C0', alpha=0.8))

ax1.text(0.99, 0.97, 'Blauw label = wisselingen per zetel per jaar (ratio)',
         transform=ax1.transAxes, fontsize=9, ha='right', va='top',
         style='italic', color='#1565C0')

plt.tight_layout()
plt.show()

# Gold Star Schema — Tweede Kamer Database

Dit is het **Gold-layer stermodel**: de brug-tabel `fractie_zetel` is geflattened,
zodat `dim_fractie` direct op de feit-tabel aansluit. De feit-tabel bevat één rij
per zetelwisseling-event (aanmelding of vertrek) met voorgecomputeerde afgeleide kolommen.
Kopieer de code hieronder naar [mermaid.live](https://mermaid.live) om het stermodel te visualiseren.

```mermaid
erDiagram
    FACT_ZETELWISSELING ||--o{ DIM_PERSOON : "persoon_id"
    FACT_ZETELWISSELING }o--|| DIM_FRACTIE : "fractie_id"
    FACT_ZETELWISSELING }o--|| DIM_TIJD : "datum_id"

    FACT_ZETELWISSELING {
        BIGINT event_id PK
        VARCHAR persoon_id FK
        VARCHAR fractie_id FK
        DATE datum_id FK
        VARCHAR event_type "aanmelding|vertrek"
        VARCHAR fractie_norm "PRO normalisatie"
        INTEGER jaar
        DOUBLE jaren_bestaan "op moment van event"
        INTEGER zetels_op_datum
    }

    DIM_PERSOON {
        VARCHAR id PK
        VARCHAR roepnaam
        VARCHAR voornamen
        VARCHAR tussenvoegsel
        VARCHAR achternaam
        VARCHAR geslacht
        DATE geboortedatum
        VARCHAR fractielabel
    }

    DIM_FRACTIE {
        VARCHAR id PK
        VARCHAR afkorting
        VARCHAR naam_nl
        VARCHAR fractie_norm "PRO normalisatie"
        INTEGER aantal_zetels
        DATE datum_actief
        DATE datum_inactief
        DOUBLE jaren_bestaan
    }

    DIM_TIJD {
        DATE datum_id PK
        INTEGER jaar
        INTEGER maand
        INTEGER kwartaal
        VARCHAR vergaderjaar
        VARCHAR maand_naam
    }
```

**Rol van elke tabel in het stermodel:**

| Tabel | Rol | Omschrijving |
| --- | --- | --- |
| `FACT_ZETELWISSELING` | **Feit (center)** | Één rij per zetelwisseling-event met meetwaarden (`zetels_op_datum`, `jaren_bestaan`) |
| `DIM_PERSOON` | **Dimensie** | Wie — persoonsgegevens (naam, geslacht, geboortedatum) |
| `DIM_FRACTIE` | **Dimensie** | Welke fractie — partijgegevens (afkorting, naam, zetels, bestaansduur) |
| `DIM_TIJD` | **Dimensie** | Wanneer — datumattributen (jaar, kwartaal, maand, vergaderjaar) |

**Verschillen met het Silver-model:**
- De brug-tabel `fractie_zetel` is verdwenen — `DIM_FRACTIE` koppelt direct aan de feit-tabel via `fractie_id`.
- De feit-tabel heet `FACT_ZETELWISSELING` (één rij per event, niet per seat-toewijzing).
- Een `DIM_TIJD` dimensie is toegevoegd voor tijd-gebaseerde analyses (jaar, kwartaal, vergaderjaar).
- Afgeleide kolommen (`fractie_norm`, `jaren_bestaan`, `zetels_op_datum`) zijn voorgecomputeerd in Gold.
- Tabelnamen volgen de `dim_` / `fact_` conventie.

In [ ]:
# Create Gold-layer tables from the Silver schema
# Matches the star schema from the Mermaid diagram above
# Creates a separate DuckDB file for the Gold layer, attaching the source as read-only

gold_db_path = database_path.parent / "tweedekamer_gold.duckdb"
write_conn = duckdb.connect(str(gold_db_path))
write_conn.execute(f"ATTACH '{database_path}' AS src (READ_ONLY)")

def gold_query_df(sql):
    """Execute a SQL query on the Gold connection and return a DataFrame."""
    return write_conn.execute(sql).fetchdf()

# Create gold schema
write_conn.execute("CREATE SCHEMA IF NOT EXISTS gold")

# 1. dim_persoon
write_conn.execute("""
CREATE OR REPLACE TABLE gold.dim_persoon AS
SELECT
    id,
    roepnaam,
    voornamen,
    tussenvoegsel,
    achternaam,
    geslacht,
    geboortedatum,
    fractielabel
FROM src.silver.persoon
""")
print(f"dim_persoon:  {write_conn.execute('SELECT COUNT(*) FROM gold.dim_persoon').fetchone()[0]:,} rows")

# 2. dim_fractie (with PRO normalisation and jaren_bestaan)
write_conn.execute("""
CREATE OR REPLACE TABLE gold.dim_fractie AS
SELECT
    id,
    afkorting,
    naam_nl,
    CASE
        WHEN afkorting IN ('PvdA', 'GL', 'GroenLinks-PvdA', 'PRO') THEN 'PRO'
        ELSE afkorting
    END AS fractie_norm,
    aantal_zetels,
    datum_actief,
    datum_inactief,
    ROUND(
        GREATEST(
            DATE_DIFF('day', datum_actief, COALESCE(datum_inactief, DATE '2026-10-10')) / 365.25,
            0.01
        ), 1
    ) AS jaren_bestaan
FROM src.silver.fractie
""")
print(f"dim_fractie:  {write_conn.execute('SELECT COUNT(*) FROM gold.dim_fractie').fetchone()[0]:,} rows")

# 3. dim_tijd (date dimension 2000–2026)
write_conn.execute("""
CREATE OR REPLACE TABLE gold.dim_tijd AS
SELECT
    d::DATE AS datum_id,
    EXTRACT(YEAR FROM d)::INTEGER AS jaar,
    EXTRACT(MONTH FROM d)::INTEGER AS maand,
    EXTRACT(QUARTER FROM d)::INTEGER AS kwartaal,
    CASE
        WHEN EXTRACT(MONTH FROM d) >= 9
        THEN EXTRACT(YEAR FROM d)::VARCHAR || '-' || (EXTRACT(YEAR FROM d) + 1)::VARCHAR
        ELSE (EXTRACT(YEAR FROM d) - 1)::VARCHAR || '-' || EXTRACT(YEAR FROM d)::VARCHAR
    END AS vergaderjaar,
    CASE EXTRACT(MONTH FROM d)
        WHEN 1 THEN 'januari' WHEN 2 THEN 'februari' WHEN 3 THEN 'maart'
        WHEN 4 THEN 'april' WHEN 5 THEN 'mei' WHEN 6 THEN 'juni'
        WHEN 7 THEN 'juli' WHEN 8 THEN 'augustus' WHEN 9 THEN 'september'
        WHEN 10 THEN 'oktober' WHEN 11 THEN 'november' WHEN 12 THEN 'december'
    END AS maand_naam
FROM generate_series(DATE '2000-01-01', DATE '2026-12-31', INTERVAL '1 day') AS t(d)
""")
print(f"dim_tijd:     {write_conn.execute('SELECT COUNT(*) FROM gold.dim_tijd').fetchone()[0]:,} rows")

# 4. fact_zetelwisseling (flattened: one row per event, fractie_id direct, no bridge)
write_conn.execute("""
CREATE OR REPLACE TABLE gold.fact_zetelwisseling AS
WITH aanmeldingen AS (
    SELECT
        fzp.persoon_id,
        f.id AS fractie_id,
        fzp.van AS datum_id,
        'aanmelding' AS event_type,
        CASE
            WHEN f.afkorting IN ('PvdA', 'GL', 'GroenLinks-PvdA', 'PRO') THEN 'PRO'
            ELSE f.afkorting
        END AS fractie_norm,
        EXTRACT(YEAR FROM fzp.van)::INTEGER AS jaar,
        GREATEST(
            DATE_DIFF('day', f.datum_actief, COALESCE(f.datum_inactief, DATE '2026-10-10')) / 365.25,
            0.01
        ) AS jaren_bestaan
    FROM src.silver.fractie_zetel_persoon fzp
    JOIN src.silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
    JOIN src.silver.fractie f ON f.id = fz.fractie_id
    WHERE fzp.van IS NOT NULL
),
vertrekken AS (
    SELECT
        fzp.persoon_id,
        f.id AS fractie_id,
        fzp.tot_en_met AS datum_id,
        'vertrek' AS event_type,
        CASE
            WHEN f.afkorting IN ('PvdA', 'GL', 'GroenLinks-PvdA', 'PRO') THEN 'PRO'
            ELSE f.afkorting
        END AS fractie_norm,
        EXTRACT(YEAR FROM fzp.tot_en_met)::INTEGER AS jaar,
        GREATEST(
            DATE_DIFF('day', f.datum_actief, COALESCE(f.datum_inactief, DATE '2026-10-10')) / 365.25,
            0.01
        ) AS jaren_bestaan
    FROM src.silver.fractie_zetel_persoon fzp
    JOIN src.silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
    JOIN src.silver.fractie f ON f.id = fz.fractie_id
    WHERE fzp.tot_en_met IS NOT NULL
),
all_events AS (
    SELECT * FROM aanmeldingen
    UNION ALL
    SELECT * FROM vertrekken
)
SELECT
    ROW_NUMBER() OVER (ORDER BY datum_id)::BIGINT AS event_id,
    persoon_id,
    fractie_id,
    datum_id,
    event_type,
    fractie_norm,
    jaar,
    ROUND(jaren_bestaan, 1) AS jaren_bestaan,
    -- zetels_op_datum: count active members in this fractie on the event date
    (
        SELECT COUNT(DISTINCT fzp2.persoon_id)
        FROM src.silver.fractie_zetel_persoon fzp2
        JOIN src.silver.fractie_zetel fz2 ON fz2.id = fzp2.fractie_zetel_id
        WHERE fz2.fractie_id = all_events.fractie_id
          AND fzp2.van <= all_events.datum_id
          AND (fzp2.tot_en_met IS NULL OR fzp2.tot_en_met >= all_events.datum_id)
    ) AS zetels_op_datum
FROM all_events
ORDER BY datum_id
""")
print(f"fact_zetelwisseling: {write_conn.execute('SELECT COUNT(*) FROM gold.fact_zetelwisseling').fetchone()[0]:,} rows")

# Show sample data from each Gold table
print("\n" + "="*60)
print("Sample: gold.dim_persoon (5 rows)")
display(gold_query_df("SELECT * FROM gold.dim_persoon LIMIT 5"))

print("\n" + "="*60)
print("Sample: gold.dim_fractie (5 rows)")
display(gold_query_df("SELECT * FROM gold.dim_fractie LIMIT 5"))

print("\n" + "="*60)
print("Sample: gold.dim_tijd (5 rows)")
display(gold_query_df("SELECT * FROM gold.dim_tijd LIMIT 5"))

print("\n" + "="*60)
print("Sample: gold.fact_zetelwisseling (10 rows)")
display(gold_query_df("SELECT * FROM gold.fact_zetelwisseling LIMIT 10"))

# Verify the star schema works: same zetelwisselingen per fractie as the EDA
print("\n" + "="*60)
print("Verification: zetelwisselingen per fractie (2020–2026) from Gold tables")
gold_check = gold_query_df("""
SELECT
    f.fractie_norm AS fractie,
    COUNT(*) AS zetelwisselingen,
    SUM(CASE WHEN fz.event_type = 'aanmelding' THEN 1 ELSE 0 END) AS aanmeldingen,
    SUM(CASE WHEN fz.event_type = 'vertrek' THEN 1 ELSE 0 END) AS vertrekken
FROM gold.fact_zetelwisseling fz
JOIN gold.dim_fractie f ON f.id = fz.fractie_id
WHERE fz.jaar BETWEEN 2020 AND 2026
GROUP BY f.fractie_norm
ORDER BY zetelwisselingen DESC
LIMIT 10
""")
display(gold_check)

print(f"\nGold tables saved to: {gold_db_path}")
# Keep write_conn open so subsequent cells can query the Gold tables